In [1]:
# This script:
# 1. Loads the Step 2 final clean dataset
# 2. Drops BIRTHMO (17.6% missing, redundant given NACCAGEB/BIRTHYR)
# 3. Handles INVISITS/INCALLS missingness using INLIVWTH context
# 4. Imputes remaining low-missingness variables
# 5. Encodes nominal categorical variables
# 6. Engineers PACK_YEARS feature
# 7. Produces the final feature matrix X and target vector y
 
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

In [2]:
INPUT_PATH = r"C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs\step2_final_clean.csv"
OUTPUT_DIR = r"C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs"
ID_COL     = "NACCID"
LABEL_COL  = "dementia_binary"
 
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Loaded: {df.shape}")

Loaded: (52537, 23)


In [3]:
 #1. Drop BIRTHMO
#
# BIRTHMO has 17.6% missingness and adds negligible signal given that NACCAGEB
# (baseline age in years) and BIRTHYR already capture age precisely.
# Birth month as a dementia risk predictor has no established clinical basis.
 
# %%
df = df.drop(columns=["BIRTHMO"])
print(f"Dropped BIRTHMO. Shape now: {df.shape}")

Dropped BIRTHMO. Shape now: (52537, 22)


In [4]:
# 2. INVISITS / INCALLS — context-aware imputation
#
# 56.3% of INVISITS/INCALLS are missing because the co-participant lives with
# the subject (INLIVWTH=1), making visit/call frequency questions inapplicable.
# Imputing with median would distort this — instead we assign a new ordinal
# category 7 = "lives with subject" which sits naturally above the existing
# 1–6 frequency scale (more contact than any discrete frequency).
# Remaining NaN after this step (INLIVWTH=0 but still missing) → median impute.
 
for col in ["INVISITS", "INCALLS"]:
    # Assign category 7 where co-participant lives with subject
    mask_lives_with = (df["INLIVWTH"] == 1) & (df[col].isna())
    df.loc[mask_lives_with, col] = 7
    # Remaining missing → median of non-missing, non-7 values
    median_val = df.loc[df[col] != 7, col].median()
    df[col] = df[col].fillna(median_val)
    print(f"{col} value counts after imputation:\n{df[col].value_counts().sort_index()}\n")

INVISITS value counts after imputation:
INVISITS
1.0     3441
2.0     5869
3.0     7872
4.0     2442
5.0     2551
6.0     2266
7.0    28096
Name: count, dtype: int64

INCALLS value counts after imputation:
INCALLS
1.0     8177
2.0     7678
3.0     4551
4.0     1078
5.0      823
6.0     2134
7.0    28096
Name: count, dtype: int64



In [7]:
#3. Engineer PACK_YEARS
#
# Pack-years is the standard epidemiological measure of cumulative smoking exposure:
#   PACK_YEARS = SMOKYRS × PACKSPER
# This single variable captures both duration and intensity of smoking more
# meaningfully than either component alone.
# For never-smokers (NEVER_SMOKED=1), SMOKYRS=0 and PACKSPER=0, so PACK_YEARS=0
# correctly — no special handling needed.
# Impute SMOKYRS and PACKSPER before computing (both <3% missing).
 

df["SMOKYRS"]  = df["SMOKYRS"].fillna(df["SMOKYRS"].median())
df["PACKSPER"] = df["PACKSPER"].fillna(df["PACKSPER"].median())
df["PACK_YEARS"] = df["SMOKYRS"] * df["PACKSPER"]
 
print(f"PACK_YEARS summary:\n{df['PACK_YEARS'].describe().round(2)}\n")
print(f"PACK_YEARS = 0 (never-smokers + zero-exposure): "
      f"{(df['PACK_YEARS'] == 0).sum()} rows ({(df['PACK_YEARS'] == 0).mean()*100:.1f}%)")

PACK_YEARS summary:
count    52537.00
mean        20.33
std         40.12
min          0.00
25%          0.00
50%          0.00
75%         22.00
max        360.00
Name: PACK_YEARS, dtype: float64

PACK_YEARS = 0 (never-smokers + zero-exposure): 32569 rows (62.0%)


In [8]:
# 4. Impute remaining low-missingness variables (<3%)
#
# Imputation strategy:
#   Numeric variables → median (robust to outliers)
#   Categorical variables → mode (most frequent value)
#
# Variables and their types:
#   Numeric:      NACCAGEB, EDUC, SMOKYRS (already done), PACKSPER (already done)
#   Categorical:  RACE, PRIMLANG, MARISTAT, NACCLIVS, RESIDENC, INRELTO,
#                 INLIVWTH, HANDED, HISPANIC, TOBAC30, TOBAC100
 
 
NUMERIC_IMPUTE = ["NACCAGEB", "EDUC"]
 
CATEGORICAL_IMPUTE = [
    "RACE", "PRIMLANG", "MARISTAT", "NACCLIVS", "RESIDENC",
    "INRELTO", "INLIVWTH", "HANDED", "HISPANIC", "TOBAC30", "TOBAC100"
]
 
for col in NUMERIC_IMPUTE:
    if col in df.columns and df[col].isna().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Imputed {col} with median={median_val}")
 
for col in CATEGORICAL_IMPUTE:
    if col in df.columns and df[col].isna().any():
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"Imputed {col} with mode={mode_val}")
 
print(f"\nAny remaining missing values?")
remaining = df.drop(columns=[ID_COL]).isna().sum()
remaining = remaining[remaining > 0]
print(remaining if len(remaining) > 0 else "None — all clean.")

Imputed NACCAGEB with median=72.0
Imputed EDUC with median=16.0
Imputed RACE with mode=1.0
Imputed PRIMLANG with mode=1.0
Imputed MARISTAT with mode=1.0
Imputed NACCLIVS with mode=2.0
Imputed RESIDENC with mode=1.0
Imputed INRELTO with mode=1.0
Imputed INLIVWTH with mode=1.0
Imputed HANDED with mode=2.0
Imputed HISPANIC with mode=0.0
Imputed TOBAC30 with mode=0.0
Imputed TOBAC100 with mode=0.0

Any remaining missing values?
None — all clean.


In [9]:
# 5. Encode nominal categorical variables
#
# Variables with no natural order → one-hot encoding (drop_first=True to avoid
# multicollinearity; this also makes the encoding interpretable as contrasts
# against the reference category).
#
# Variables that are already numeric ordinal or binary are left as-is:
#   Binary (0/1):    SEX, HISPANIC, NEVER_SMOKED, TOBAC30, TOBAC100, INLIVWTH
#   Ordinal numeric: NACCAGEB, BIRTHYR, EDUC, SMOKYRS, PACKSPER, PACK_YEARS,
#                    INVISITS (1-7), INCALLS (1-7)
#
# Nominal → one-hot:
#   RACE, PRIMLANG, MARISTAT, NACCLIVS, RESIDENC, INRELTO, HANDED

NOMINAL_COLS = ["RACE", "PRIMLANG", "MARISTAT", "NACCLIVS", "RESIDENC", "INRELTO", "HANDED"]
 
# Convert to string first so pd.get_dummies treats them as categorical
for col in NOMINAL_COLS:
    df[col] = df[col].astype(str)
 
df_encoded = pd.get_dummies(df, columns=NOMINAL_COLS, drop_first=True, dtype=int)
 
print(f"Shape after one-hot encoding: {df_encoded.shape}")
new_cols = [c for c in df_encoded.columns if c not in df.columns]
print(f"New one-hot columns ({len(new_cols)}): {new_cols}")

Shape after one-hot encoding: (52537, 46)
New one-hot columns (30): ['RACE_2.0', 'RACE_3.0', 'RACE_4.0', 'RACE_5.0', 'RACE_50.0', 'PRIMLANG_2.0', 'PRIMLANG_3.0', 'PRIMLANG_4.0', 'PRIMLANG_5.0', 'PRIMLANG_6.0', 'MARISTAT_2.0', 'MARISTAT_3.0', 'MARISTAT_4.0', 'MARISTAT_5.0', 'MARISTAT_6.0', 'NACCLIVS_2.0', 'NACCLIVS_3.0', 'NACCLIVS_4.0', 'NACCLIVS_5.0', 'RESIDENC_2.0', 'RESIDENC_3.0', 'RESIDENC_4.0', 'INRELTO_2.0', 'INRELTO_3.0', 'INRELTO_4.0', 'INRELTO_5.0', 'INRELTO_6.0', 'INRELTO_7.0', 'HANDED_2.0', 'HANDED_3.0']


In [10]:
# 6. Final feature matrix
 

# Separate ID and label from features
feature_cols = [c for c in df_encoded.columns if c not in [ID_COL, LABEL_COL]]
 
X = df_encoded[feature_cols]
y = df_encoded[LABEL_COL]
 
print(f"\nFinal feature matrix X: {X.shape}")
print(f"Target vector y: {y.shape}")
print(f"\nFeature list ({len(feature_cols)}):")
print(feature_cols)
 
print(f"\nClass balance:\n{(y.value_counts(normalize=True) * 100).round(1)}")
print(f"\nAny NaN in X? {X.isna().any().any()}")
print(f"Any NaN in y? {y.isna().any()}")


Final feature matrix X: (52537, 44)
Target vector y: (52537,)

Feature list (44):
['BIRTHYR', 'NACCAGEB', 'SEX', 'HISPANIC', 'EDUC', 'TOBAC30', 'TOBAC100', 'SMOKYRS', 'PACKSPER', 'INLIVWTH', 'INVISITS', 'INCALLS', 'NEVER_SMOKED', 'PACK_YEARS', 'RACE_2.0', 'RACE_3.0', 'RACE_4.0', 'RACE_5.0', 'RACE_50.0', 'PRIMLANG_2.0', 'PRIMLANG_3.0', 'PRIMLANG_4.0', 'PRIMLANG_5.0', 'PRIMLANG_6.0', 'MARISTAT_2.0', 'MARISTAT_3.0', 'MARISTAT_4.0', 'MARISTAT_5.0', 'MARISTAT_6.0', 'NACCLIVS_2.0', 'NACCLIVS_3.0', 'NACCLIVS_4.0', 'NACCLIVS_5.0', 'RESIDENC_2.0', 'RESIDENC_3.0', 'RESIDENC_4.0', 'INRELTO_2.0', 'INRELTO_3.0', 'INRELTO_4.0', 'INRELTO_5.0', 'INRELTO_6.0', 'INRELTO_7.0', 'HANDED_2.0', 'HANDED_3.0']

Class balance:
dementia_binary
1    54.7
0    45.3
Name: proportion, dtype: float64

Any NaN in X? False
Any NaN in y? False


In [11]:
# 7. Save for Step 4 (Model Development)
 
# %%
out_X      = os.path.join(OUTPUT_DIR, "step3_X.csv")
out_y      = os.path.join(OUTPUT_DIR, "step3_y.csv")
out_full   = os.path.join(OUTPUT_DIR, "step3_full.csv")
 
X.to_csv(out_X, index=False)
y.to_csv(out_y, index=False)
df_encoded[[ID_COL] + feature_cols + [LABEL_COL]].to_csv(out_full, index=False)
 
print(f"Saved:\n  {out_X}\n  {out_y}\n  {out_full}")

Saved:
  C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs\step3_X.csv
  C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs\step3_y.csv
  C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs\step3_full.csv
